In [ ]:
import zarr
import numpy as np
import tensorflow as tf
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from keras.models import Model
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import math
import numpy as np
import os
import gc
import random
import tensorflow as tf
from keras.losses import BinaryFocalCrossentropy

colors_phiSat = ['green', 'red', 'lightgray', 'blue']
my_cmap_phiSat = ListedColormap(colors_phiSat)

box_0_phiSat = mpatches.Patch(color=colors_phiSat[0], label='0: Background')
box_1_phiSat = mpatches.Patch(color=colors_phiSat[1], label='1: Burned area')
box_2_phiSat = mpatches.Patch(color=colors_phiSat[2], label='2: Clouds')
box_3_phiSat = mpatches.Patch(color=colors_phiSat[3], label='3: Water')

In [ ]:
# Load data
def get_data_unet(id_list, store):
    X_list, y_list = [], []
    for sample in id_list:
        image = np.array(zarr.open_array(store, path=f"trainval/{sample}/img", mode='r'))
        label = np.array(zarr.open_array(store, path=f"trainval/{sample}/label", mode='r'))
        
        if len(label.shape) == 3:
            label = np.argmax(label, axis=0)
            
        label = (label == 1).astype(np.uint8)
        label = np.expand_dims(label, axis=-1)

        image_reshaped = np.moveaxis(image, 0, -1)
        image_reshaped = image_reshaped.astype(np.float32) / np.max(image_reshaped)

        X_list.append(image_reshaped)
        y_list.append(label)
        
    return np.array(X_list), np.array(y_list)

def unet_generator(id_list, store, batch_size=8):
    ids = np.array(id_list) 
    
    while True:
        np.random.shuffle(ids) 
        
        for i in range(0, len(ids), batch_size):
            batch_ids = ids[i:i + batch_size]
            X_batch, y_batch = get_data_unet(batch_ids, store)
            yield X_batch, y_batch

def build_unet(input_shape=(256, 256, 7), num_classes=1): # Set num_classes=1 for binary segmentation, >1 for multiclass
    inputs = Input(input_shape)
    
    # Encoder
    c1 = Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    p1 = MaxPooling2D((2, 2))(c1)
    
    c2 = Conv2D(32, (3, 3), activation='relu', padding='same')(p1)
    p2 = MaxPooling2D((2, 2))(c2)
    
    # Bottleneck
    c3 = Conv2D(64, (3, 3), activation='relu', padding='same')(p2)
    
    # Decoder
    u4 = UpSampling2D((2, 2))(c3)
    concat4 = concatenate([u4, c2])
    c4 = Conv2D(32, (3, 3), activation='relu', padding='same')(concat4)
    
    u5 = UpSampling2D((2, 2))(c4)
    concat5 = concatenate([u5, c1]) 
    c5 = Conv2D(16, (3, 3), activation='relu', padding='same')(concat5)
    
    # Output
    outputs = Conv2D(num_classes, (1, 1), activation='sigmoid')(c5) # activation='softmax' for multiclass, 'sigmoid' for binary
    
    model = Model(inputs=[inputs], outputs=[outputs])
    return model

In [ ]:
zarr_path = "/home/laura/Scriptie/ruwe_data/burned.zarr.zip"
store_phiSat2 = zarr.ZipStore(zarr_path, mode='r')

all_keys = list(store_phiSat2.keys())
samples = list(set([key.split('/')[1] for key in all_keys if key.startswith('trainval/') and key.split('/')[1].isdigit()]))

subset_samples = samples[0:1000]

train_samples, val_samples = train_test_split(subset_samples, train_size=0.8, test_size=0.2, shuffle=False)

unet = build_unet()
unet.compile(optimizer='adam', loss=BinaryFocalCrossentropy(alpha=0.9, gamma=2.0), metrics=['accuracy']) # Use 'categorical_crossentropy' for multiclass, 'binary_crossentropy' for binary
unet.summary()

'''zarr_path = "/home/laura/Scriptie/ruwe_data/burned.zarr.zip"
store_phiSat2 = zarr.ZipStore(zarr_path, mode='r')

all_keys = list(store_phiSat2.keys())
samples = list(set([key.split('/')[1] for key in all_keys if key.startswith('trainval/') and key.split('/')[1].isdigit()]))
random.seed(42)
random.shuffle(samples)

subset_samples = samples[0:1000]

train_samples, val_samples = train_test_split(subset_samples, train_size=0.8, test_size=0.2, shuffle=False)

unet = build_unet()
unet.compile(optimizer='adam', loss=BinaryFocalCrossentropy(alpha=0.9, gamma=2.0), metrics=['accuracy']) # Use 'categorical_crossentropy' for multiclass, 'binary_crossentropy' for binary
unet.summary()'''

In [ ]:
batch_size = 3
train_gen = unet_generator(train_samples, store_phiSat2, batch_size=batch_size)
val_gen = unet_generator(val_samples, store_phiSat2, batch_size=batch_size)

steps_per_epoch = math.ceil(len(train_samples) / batch_size)
validation_steps = math.ceil(len(val_samples) / batch_size)

X_batch_test, y_batch_test = next(iter(train_gen))

print("Shape of X (Features):", X_batch_test.shape)
print("Shape of y (Labels)  :", y_batch_test.shape)
print("Datatype of y       :", y_batch_test.dtype)
print("Unique values in y  :", np.unique(y_batch_test))

unet_history = unet.fit(
    train_gen, 
    epochs=4, 
    steps_per_epoch=steps_per_epoch, 
    validation_data=val_gen, 
    validation_steps=validation_steps, 
    verbose=1
)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(unet_history.history['loss'], label='Training Loss')
plt.plot(unet_history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(unet_history.history['accuracy'], label='Training Accuracy')
plt.plot(unet_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
bewerkte_data_dir = '/home/laura/Scriptie/bewerkte_data/PhiSat2'
os.makedirs(bewerkte_data_dir, exist_ok=True)

num_sample_img = len(val_samples)
pixels_per_img = 256 * 256

y_true_full = np.zeros((num_sample_img, pixels_per_img), dtype=np.int8)
y_probs_full = np.zeros((num_sample_img, pixels_per_img), dtype=np.float16)
y_preds_full = np.zeros((num_sample_img, pixels_per_img), dtype=np.int8)

batch_size = 8
total_batches = (num_sample_img + batch_size - 1) // batch_size

for i in range(0, num_sample_img, batch_size):
    batch_ids = val_samples[i:i + batch_size]
    batch_size_actual = len(batch_ids)
    
    X_batch, y_batch = get_data_unet(batch_ids, store_phiSat2)
    
    preds_batch = unet.predict_on_batch(X_batch)
    
    start_idx = i
    end_idx = i + batch_size_actual

    y_true_full[start_idx:end_idx] = y_batch.reshape(batch_size_actual, -1).astype(np.int8)
    y_probs_full[start_idx:end_idx] = preds_batch[..., 0].reshape(batch_size_actual, -1).astype(np.float16)
    y_preds_full[start_idx:end_idx] = (preds_batch[..., 0] >= 0.5).reshape(batch_size_actual, -1).astype(np.int8)
    
    del X_batch, y_batch, preds_batch
    gc.collect()

np.save(os.path.join(bewerkte_data_dir, 'y_test_masks_unet.npy'), y_true_full.flatten())
np.save(os.path.join(bewerkte_data_dir, 'y_pred_probs_fire_unet.npy'), y_probs_full.flatten())
np.save(os.path.join(bewerkte_data_dir, 'y_pred_masks_unet.npy'), y_preds_full.flatten())

del y_true_full, y_probs_full, y_preds_full
gc.collect()